# ❤️ ECG Anomaly Detection — Phase 3: Model Training

This notebook trains the hybrid **1D-CNN + Transformer** model on the [PTB-XL dataset](https://physionet.org/content/ptb-xl/1.0.3/) (21,799 12-lead ECGs at 500 Hz) to detect 10 cardiac conditions with calibrated confidence scores.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (or A100 if you have Colab Pro)
2. **Runtime → Run all** — cells will execute sequentially
3. Allow ~20-30 min for the PTB-XL download (~3 GB), then ~2-4 hours for training

Checkpoints are saved to Google Drive so training can be resumed if the session disconnects.

---
| Condition | PTB-XL SCP codes |
|-----------|------------------|
| Normal Sinus Rhythm | NORM |
| Atrial Fibrillation | AFIB, AFLT |
| ST Elevation (STEMI) | AMI, IMI, STE\_ ... |
| Left Bundle Branch Block | LBBB, CLBBB |
| Right Bundle Branch Block | RBBB, CRBBB, IRBBB |
| Left Ventricular Hypertrophy | LVH |
| Bradycardia | SBRAD, BRAD |
| Tachycardia | STACH, SVTACH |
| First Degree AV Block | 1AVB |
| Premature Ventricular Contraction | PVC, VPCS |

## 1. Mount Google Drive & set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit these paths if you want to store data/checkpoints elsewhere ──────
BASE_DIR       = '/content/drive/MyDrive/ecg_project'
DATA_DIR       = f'{BASE_DIR}/ptbxl'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
LOG_DIR        = f'{BASE_DIR}/logs'
EVAL_DIR       = f'{BASE_DIR}/eval_output'
# ─────────────────────────────────────────────────────────────────────────

for d in [BASE_DIR, DATA_DIR, CHECKPOINT_DIR, LOG_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

print('Paths ready:')
for label, path in [('data', DATA_DIR), ('checkpoints', CHECKPOINT_DIR),
                     ('logs', LOG_DIR), ('eval', EVAL_DIR)]:
    print(f'  {label:15s}: {path}')

Mounted at /content/drive
Paths ready:
  data           : /content/drive/MyDrive/ecg_project/ptbxl
  checkpoints    : /content/drive/MyDrive/ecg_project/checkpoints
  logs           : /content/drive/MyDrive/ecg_project/logs
  eval           : /content/drive/MyDrive/ecg_project/eval_output


In [3]:
# ── Session Recovery: rebuild /content/phase3 from Drive if missing ────────
import os, sys, zipfile, shutil
from pathlib import Path

sys.path.insert(0, '/content/phase3')

phase3_ok = Path('/content/phase3/ecg_model/dataset.py').exists()

if phase3_ok:
    print('✓ /content/phase3 already present — nothing to do')
else:
    print('/content/phase3 is missing (session reset). Rebuilding...')

    # Option A: you saved a zip to Drive previously (fastest)
    zip_candidates = [
        f'{BASE_DIR}/phase3_signal_preprocessing.zip',
        f'{BASE_DIR}/phase3.zip',
    ]
    restored = False
    for zp in zip_candidates:
        if os.path.exists(zp):
            print(f'  Found zip: {zp} — extracting...')
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall('/content/')
            restored = True
            print('  ✓ Extracted from Drive zip')
            break

    if not restored:
        print('  No zip found — re-running the %%writefile cells is needed.')
        print('  TIP: After rebuilding once, save a zip to Drive so recovery')
        print('  is instant next time:')
        print('    import shutil')
        print('    shutil.make_archive(f"{BASE_DIR}/phase3", "zip", "/content/phase3")')
        print('    print("Saved phase3.zip to Drive")')
        raise RuntimeError('Re-run cells 3-4 (the %%writefile / model code cells) first.')

# ── Re-apply the _time_warp fix every session (patch on disk + in memory) ──
import numpy as np

def _time_warp_fixed(signal, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = signal.shape
    n_anchors = 6
    xp     = np.linspace(0, n_samples - 1, n_anchors)
    shifts = rng.uniform(-max_warp * n_samples,
                          max_warp * n_samples, n_anchors)
    shift_curve = np.interp(np.arange(n_samples), xp, shifts)
    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)
    return np.stack([
        np.interp(new_idx, old_idx, signal[i]) for i in range(n_leads)
    ], axis=0).astype(signal.dtype)

dataset_path = '/content/phase3/ecg_model/dataset.py'
with open(dataset_path, 'r') as f:
    src = f.read()
start = src.find('\ndef _time_warp')
end   = src.find('\ndef ', start + 1)
if end == -1: end = len(src)
new_fn = '''
def _time_warp(signal, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = signal.shape
    n_anchors = 6
    xp     = np.linspace(0, n_samples - 1, n_anchors)
    shifts = rng.uniform(-max_warp * n_samples,
                          max_warp * n_samples, n_anchors)
    shift_curve = np.interp(np.arange(n_samples), xp, shifts)
    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)
    return np.stack([
        np.interp(new_idx, old_idx, signal[i]) for i in range(n_leads)
    ], axis=0).astype(signal.dtype)

'''
with open(dataset_path, 'w') as f:
    f.write(src[:start] + new_fn + src[end:])

import ecg_model.dataset as ds_mod
ds_mod._time_warp = _time_warp_fixed
print('✓ _time_warp patch applied')

# ── Verify checkpoint from last session ────────────────────────────────────
ckpt = Path(CHECKPOINT_DIR) / 'best_model.pt'
if ckpt.exists():
    import torch
    meta = torch.load(ckpt, map_location='cpu')
    last_epoch = meta.get('epoch', '?')
    last_auc   = meta.get('val_auc', '?')
    print(f'✓ Checkpoint found: epoch {last_epoch}, val_AUC {last_auc:.4f}')
    print(f'  Training will early-stop if it cannot beat this AUC.')
else:
    print('  No checkpoint yet — training will start fresh.')

/content/phase3 is missing (session reset). Rebuilding...
  No zip found — re-running the %%writefile cells is needed.
  TIP: After rebuilding once, save a zip to Drive so recovery
  is instant next time:
    import shutil
    shutil.make_archive(f"{BASE_DIR}/phase3", "zip", "/content/phase3")
    print("Saved phase3.zip to Drive")


RuntimeError: Re-run cells 3-4 (the %%writefile / model code cells) first.

## 2. Install dependencies

In [4]:
# wfdb is the only package Colab doesn't have pre-installed.
!pip install -q wfdb

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 97.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
PyTorch 2.10.0+cu

## 3. Clone / upload the Phase 3 code

In [5]:
# Option A: if you uploaded phase3.zip to Drive, unzip it:
# !unzip -q /content/drive/MyDrive/phase3.zip -d /content/

# Option B: if you pushed the code to a GitHub repo:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/phase3

# Option C (default): write the code directly from this notebook.
# The cells below write each module to disk so you don't need an external repo.
import os, sys
os.makedirs('/content/phase3/ecg_model', exist_ok=True)
sys.path.insert(0, '/content/phase3')
print('Code directory ready at /content/phase3')

Code directory ready at /content/phase3


In [6]:
%%writefile /content/phase3/ecg_model/__init__.py
"""ecg_model — Phase 3 ML training package."""
from .model import ECGAnomalyDetector, ModelConfig, CONDITION_NAMES, N_CONDITIONS, build_model, load_checkpoint
from .dataset import PTBXLDataset, load_ptbxl, make_dataloaders, download_ptbxl
from .train import TrainConfig, train
from .evaluate import evaluate, plot_training_history

Writing /content/phase3/ecg_model/__init__.py


In [7]:
# ── Write model.py ────────────────────────────────────────────────────────
MODEL_CODE = '''
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn

CONDITION_NAMES = (
    "Normal Sinus Rhythm",
    "Atrial Fibrillation",
    "ST Elevation",
    "Left Bundle Branch Block",
    "Right Bundle Branch Block",
    "Left Ventricular Hypertrophy",
    "Bradycardia",
    "Tachycardia",
    "First Degree AV Block",
    "Premature Ventricular Contraction",
)
N_CONDITIONS = len(CONDITION_NAMES)

@dataclass
class ModelConfig:
    n_leads: int = 12
    n_samples: int = 5000
    n_conditions: int = N_CONDITIONS
    cnn_channels: tuple = (32, 64, 128, 128)
    cnn_kernel_sizes: tuple = (7, 5, 5, 3)
    cnn_pool_sizes: tuple = (2, 2, 2, 2)
    cnn_dropout: float = 0.1
    d_model: int = 128
    n_heads: int = 4
    n_transformer_layers: int = 2
    dim_feedforward: int = 512
    transformer_dropout: float = 0.1
    head_hidden: int = 256
    head_dropout: float = 0.3

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, pool, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(pool), nn.Dropout(dropout),
        )
    def forward(self, x): return self.block(x)

class LeadEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        chs = [1] + list(cfg.cnn_channels)
        self.blocks = nn.ModuleList([
            ConvBlock(chs[i], chs[i+1], cfg.cnn_kernel_sizes[i],
                      cfg.cnn_pool_sizes[i], cfg.cnn_dropout)
            for i in range(len(cfg.cnn_channels))
        ])
    def forward(self, x):
        B, L, N = x.shape
        x = x.reshape(B*L, 1, N)
        for blk in self.blocks: x = blk(x)
        _, C, T = x.shape
        return x.reshape(B, L, C, T)

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, n_leads, max_frames, d_model):
        super().__init__()
        self.lead_emb = nn.Embedding(n_leads, d_model)
        self.time_emb = nn.Embedding(max_frames, d_model)
    def forward(self, x, n_leads, n_frames):
        dev = x.device
        leads = torch.arange(n_leads, device=dev).repeat_interleave(n_frames)
        times = torch.arange(n_frames, device=dev).repeat(n_leads)
        return x + self.lead_emb(leads) + self.time_emb(times)

class ECGAnomalyDetector(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        if cfg is None: cfg = ModelConfig()
        self.cfg = cfg
        self.lead_encoder = LeadEncoder(cfg)
        t_prime = cfg.n_samples
        for p in cfg.cnn_pool_sizes: t_prime = t_prime // p
        self.t_prime = t_prime
        cnn_out_ch = cfg.cnn_channels[-1]
        self.input_proj = nn.Identity() if cnn_out_ch == cfg.d_model else nn.Linear(cnn_out_ch, cfg.d_model)
        self.pos_enc = LearnablePositionalEncoding(cfg.n_leads, self.t_prime + 32, cfg.d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.n_heads,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.transformer_dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_transformer_layers,
                                                  enable_nested_tensor=False)
        self.classifier = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.head_hidden),
            nn.LayerNorm(cfg.head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(cfg.head_hidden, cfg.n_conditions),
        )
        self.register_buffer("temperature", torch.ones(1))
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight); m.bias is not None and nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        B, L, N = x.shape
        feats = self.lead_encoder(x)
        _, _, C, T = feats.shape
        feats = feats.permute(0,1,3,2).reshape(B, L*T, C)
        feats = self.input_proj(feats)
        feats = self.pos_enc(feats, n_leads=L, n_frames=T)
        feats = self.transformer(feats)
        feats = feats.mean(dim=1)
        return self.classifier(feats)
    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.sigmoid(self.forward(x) / self.temperature.clamp(min=0.05))
    def parameter_count(self):
        return {"total": sum(p.numel() for p in self.parameters()),
                "trainable": sum(p.numel() for p in self.parameters() if p.requires_grad)}

class TemperatureScaler:
    def __init__(self, model): self.model = model
    def fit(self, logits, labels, lr=0.01, max_iter=50):
        temperature = nn.Parameter(self.model.temperature.clone().requires_grad_(True))
        opt = torch.optim.LBFGS([temperature], lr=lr, max_iter=max_iter)
        crit = nn.BCEWithLogitsLoss()
        def step():
            opt.zero_grad()
            loss = crit(logits / temperature.clamp(min=0.05), labels)
            loss.backward()
            return loss
        opt.step(step)
        self.model.temperature.copy_(temperature.detach().clamp(min=0.05))
        return crit(logits / self.model.temperature, labels).item()

def build_model(cfg=None, device=None):
    m = ECGAnomalyDetector(cfg)
    if device: m = m.to(device)
    return m

def load_checkpoint(path, device="cpu", cfg=None):
    from dataclasses import fields
    ckpt = torch.load(path, map_location=device)
    cfg = cfg or ModelConfig(**{k: v for k,v in ckpt.get("cfg", {}).items()
                                if k in {f.name for f in fields(ModelConfig)}})
    model = ECGAnomalyDetector(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    return model
'''
with open('/content/phase3/ecg_model/model.py', 'w') as f:
    f.write(MODEL_CODE)
print('model.py written')

model.py written


In [8]:
# Copy the dataset.py, train.py, evaluate.py from the uploaded zip (Option A)
# OR paste them inline below (Option C).
# If you used the zip upload, uncomment and skip the %%writefile cells:

# import shutil
# for fname in ['dataset.py', 'train.py', 'evaluate.py']:
#     shutil.copy(f'/content/phase3_extracted/ecg_model/{fname}',
#                 f'/content/phase3/ecg_model/{fname}')

# Otherwise — the ZIP download cell below fetches the full package from Drive:
import shutil, zipfile, os
zip_path = f'{BASE_DIR}/phase3_signal_preprocessing.zip'  # uploaded earlier
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('Extracted zip from Drive')
else:
    print('No zip found — make sure dataset.py, train.py, evaluate.py '
          'are in /content/phase3/ecg_model/ before proceeding.')
    print('You can upload them via the Files pane on the left, or use %%writefile cells.')

No zip found — make sure dataset.py, train.py, evaluate.py are in /content/phase3/ecg_model/ before proceeding.
You can upload them via the Files pane on the left, or use %%writefile cells.


## 4a. Set up Kaggle credentials

Upload your `kaggle.json` API token (from https://www.kaggle.com/settings → *Create New Token*).
This is required to download PTB-XL via the Kaggle dataset mirror.

In [9]:
import json

kaggle_dict = {
    "username": "Singhbrothers17",
    "key": "KGAT_69af3fc9163066f76c25f7b51b11a9c9"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_dict, f)

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured ✓')

Kaggle credentials configured ✓


## 4b. Download PTB-XL via Kaggle (~3 GB)

Uses the Kaggle API mirror (`khyeh0719/ptb-xl-dataset`) — much more reliable than PhysioNet wget.
Skips automatically if the dataset is already present in Drive.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DATA_DIR = '/content/ptbxl'
TMP_DIR = '/content/ptbxl_tmp'
ZIP_PATH = '/content/ptb-xl-dataset.zip'

meta_path = Path(DATA_DIR) / 'ptbxl_database.csv'

if meta_path.exists():
    print(f'PTB-XL already present at {DATA_DIR} — skipping download.')

else:
    print('Downloading PTB-XL (~3GB)...')
    !pip install -q kaggle
    !kaggle datasets download -d khyeh0719/ptb-xl-dataset -p /content --force

    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(TMP_DIR)

    # 🔥 Directly point to known folder (no rglob)
    extracted_dir = Path(TMP_DIR) / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'

    if not extracted_dir.exists():
        raise FileNotFoundError("Expected dataset folder not found after extraction.")

    print(f'Moving files to {DATA_DIR} ...')
    os.makedirs(DATA_DIR, exist_ok=True)

    for item in extracted_dir.iterdir():
        dest = Path(DATA_DIR) / item.name
        if not dest.exists():
            shutil.move(str(item), str(dest))

    # Cleanup
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)

    print('✅ Dataset ready!')

# Verify
import pandas as pd

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f'\nPTB-XL loaded: {len(df):,} records')
    print(df[['patient_id', 'age', 'sex']].head())
else:
    print('❌ Still not found — something is off.')

Dataset URL: https://www.kaggle.com/datasets/khyeh0719/ptb-xl-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
 28% 486M/1.72G [00:30<01:14, 17.9MB/s]

## 5. Explore the dataset & class distribution

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/phase3/')

from ecg_model.dataset import scp_to_labels, load_ptbxl, PTBXLDataset
from ecg_model.model import CONDITION_NAMES

train_ds, val_ds, test_ds = load_ptbxl(DATA_DIR)
print(f'Split sizes — train: {len(train_ds):,}  val: {len(val_ds):,}  test: {len(test_ds):,}')

label_matrix = train_ds.label_matrix
counts = label_matrix.sum(axis=0)
total = len(label_matrix)

print('\nClass distribution (training set):')
print(f'{"Condition":<44} Count    %')
for name, c in zip(CONDITION_NAMES, counts):
    print(f'  {name:<42} {int(c):5d}  {100*c/total:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ecc71' if c/total > 0.15 else '#f39c12' if c/total > 0.05 else '#e74c3c'
          for c in counts]
ax.barh([n.replace('Left ', 'L-').replace('Right ', 'R-') for n in CONDITION_NAMES],
        counts, color=colors)
ax.set_xlabel('Number of records')
ax.set_title('PTB-XL Training Set — Class Distribution')
for i, v in enumerate(counts):
    ax.text(v + 50, i, f'{v:.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Diagnostic: check what the dataset actually loaded ────────────────────
import wfdb, pandas as pd, os
from pathlib import Path

df = pd.read_csv(Path(DATA_DIR) / 'ptbxl_database.csv')
rec = df.iloc[0]

# Check which record paths exist
for col in ['filename_lr', 'filename_hr']:
    full = Path(DATA_DIR) / rec[col]
    exists = full.with_suffix('.hea').exists()
    print(f"{col}: {rec[col]}  →  exists={exists}")

# Read one record directly and inspect shape
for col in ['filename_hr', 'filename_lr']:
    full = str(Path(DATA_DIR) / rec[col])
    try:
        sig, meta = wfdb.rdsamp(full)
        print(f"\n{col}: shape={sig.shape}, fs={meta['fs']} Hz")
        break
    except Exception as e:
        print(f"\n{col}: FAILED — {e}")

## 6. Inspect a sample ECG

In [ ]:
# ── Section 6: Inspect a sample ECG (reads wfdb directly — no resampling) ──
import wfdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv(Path(DATA_DIR) / 'ptbxl_database.csv')

# Auto-pick whichever resolution is actually present
rec = df.iloc[0]
sig, meta = None, None
for col in ['filename_hr', 'filename_lr']:
    path = str(Path(DATA_DIR) / rec[col])
    try:
        sig, meta = wfdb.rdsamp(path)
        print(f"Loaded via '{col}': shape={sig.shape}, fs={meta['fs']} Hz")
        break
    except Exception as e:
        print(f"'{col}' failed: {e}")

if sig is None:
    raise RuntimeError("Could not load any record — check DATA_DIR paths above.")

# Z-score each lead for display
sig = (sig - sig.mean(axis=0)) / (sig.std(axis=0) + 1e-8)
t = np.arange(sig.shape[0]) / meta['fs']

lead_names = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()
for i, ax in enumerate(axes):
    ax.plot(t, sig[:, i], linewidth=0.6, color='#2c3e50')
    ax.set_ylabel(lead_names[i], fontsize=9, rotation=0, labelpad=25)
    ax.set_ylim(-3, 3)
    ax.tick_params(labelsize=7)
    ax.axhline(0, color='#bdc3c7', linewidth=0.4)
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Sample ECG — patient {rec["patient_id"]} | {meta["fs"]} Hz',
             fontsize=11)
plt.tight_layout()
plt.show()

## 7. Build model & verify architecture

In [ ]:
import torch
from ecg_model.model import build_model, ModelConfig, CONDITION_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

cfg = ModelConfig()  # defaults: 4 CNN blocks, 2 Transformer layers, 4 heads
model = build_model(cfg, device=str(device))

counts = model.parameter_count()
print(f'\nModel architecture:')
print(f'  Total parameters:     {counts["total"]:,}')
print(f'  Trainable parameters: {counts["trainable"]:,}')
print(f'  CNN channels:         {cfg.cnn_channels}')
print(f'  Transformer heads:    {cfg.n_heads}')
print(f'  Transformer layers:   {cfg.n_transformer_layers}')
print(f'  Tokens per forward:   {cfg.n_leads * model.t_prime}')

# Forward pass smoke test.
dummy = torch.randn(2, 12, 5000).to(device)
with torch.no_grad():
    logits = model(dummy)
    probs = model.predict_proba(dummy)
print(f'\nForward pass OK:')
print(f'  Input:   {tuple(dummy.shape)}')
print(f'  Logits:  {tuple(logits.shape)}')
print(f'  Probs range: [{probs.min().item():.3f}, {probs.max().item():.3f}]')
assert probs.shape == (2, len(CONDITION_NAMES))
assert probs.min().item() >= 0.0 and probs.max().item() <= 1.0
print('  Assertions passed ✓')

In [ ]:
# ── Patch dataset to handle 100 Hz records ────────────────────────────────
# If the diagnostic showed filename_lr / 100 Hz, the dataset.py is loading
# the wrong column. Override n_samples to match.
import wfdb, pandas as pd
from pathlib import Path

df = pd.read_csv(Path(DATA_DIR) / 'ptbxl_database.csv')
rec = df.iloc[0]

# Detect actual sample count
for col in ['filename_hr', 'filename_lr']:
    try:
        sig, meta = wfdb.rdsamp(str(Path(DATA_DIR) / rec[col]))
        ACTUAL_FS       = meta['fs']          # 100 or 500
        ACTUAL_SAMPLES  = sig.shape[0]        # 1000 or 5000
        ACTUAL_COL      = col
        print(f"Using '{col}': {ACTUAL_FS} Hz, {ACTUAL_SAMPLES} samples")
        break
    except:
        continue

## 8. Configure & run training

In [ ]:
from ecg_model.train import TrainConfig, train as run_training

from ecg_model.model import ModelConfig
cfg = TrainConfig(
    data_dir        = DATA_DIR,
    checkpoint_dir  = CHECKPOINT_DIR,
    log_dir         = LOG_DIR,
    epochs          = 50,
    batch_size      = 64,
    num_workers     = 2,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    warmup_epochs   = 3,
    patience        = 7,
    pos_weight_cap  = 20.0,
    seed            = 42,
    model_cfg       = ModelConfig(n_samples=ACTUAL_SAMPLES),  # ← add this line
)

print('Training configuration:')
for k, v in vars(cfg).items():
    if k != 'model_cfg':
        print(f'  {k:<22}: {v}')
print()
print('Starting training... (checkpoints saved to Drive on every val-AUC improvement)')
print('The session can disconnect and resume — the best checkpoint persists in Drive.\n')

In [ ]:
# ── Bulletproof in-memory fix: replace _time_warp with a clean version ────
# This bypasses whatever is in dataset.py on disk. DataLoader workers fork
# from this process, so they inherit this binding.

import numpy as np
import ecg_model.dataset as D
import inspect

def _time_warp_clean(signal, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = signal.shape
    n_anchors = 6
    xp_anchors  = np.linspace(0, n_samples - 1, n_anchors)
    shifts      = rng.uniform(-max_warp * n_samples, max_warp * n_samples, n_anchors)
    shift_curve = np.interp(np.arange(n_samples), xp_anchors, shifts)

    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)

    out = np.empty_like(signal, dtype=np.float64)
    for i in range(n_leads):
        out[i] = np.interp(new_idx, old_idx, signal[i])
    return out.astype(signal.dtype)

D._time_warp = _time_warp_clean

# Confirm what Python now sees
print("Module on disk :", D.__file__)
print("_time_warp is now:")
print(inspect.getsource(D._time_warp))

# Smoke test — this must succeed before we touch training
sig = np.random.randn(12, 5000).astype(np.float32)
rng = np.random.default_rng(0)
out = D._time_warp(sig, rng)
assert out.shape == sig.shape, f"Shape mismatch: {out.shape} vs {sig.shape}"
print(f"✓ Smoke test passed: in={sig.shape}, out={out.shape}")

In [ ]:
# ── Patch _time_warp in dataset.py to fix the interp length mismatch ──────
import re

dataset_path = '/content/phase3/ecg_model/dataset.py'

with open(dataset_path, 'r') as f:
    src = f.read()

# Show the current broken function so you can confirm what's there
start = src.find('def _time_warp')
print("CURRENT _time_warp:\n")
print(src[start:start+600])

In [ ]:
# ── Apply the fix ──────────────────────────────────────────────────────────
dataset_path = '/content/phase3/ecg_model/dataset.py'

with open(dataset_path, 'r') as f:
    src = f.read()

# Find and replace the entire _time_warp function with a correct version
old_fn_start = src.find('\ndef _time_warp')
old_fn_end   = src.find('\ndef ', old_fn_start + 1)   # next function
if old_fn_end == -1:
    old_fn_end = len(src)

old_fn = src[old_fn_start:old_fn_end]
print("Replacing:\n", old_fn[:400], "\n...")

new_fn = '''
def _time_warp(signal, rng, fs=500, max_warp=0.05):
    """
    Apply smooth time-warp augmentation to an ECG signal.
    signal: (n_leads, n_samples)
    Returns signal of the same shape.
    """
    n_leads, n_samples = signal.shape

    # Build a smooth warp map using a few random anchor points
    n_anchors = 6
    anchors   = np.sort(rng.uniform(0, n_samples, n_anchors))
    shifts    = rng.uniform(-max_warp * n_samples,
                             max_warp * n_samples, n_anchors)

    # xp / fp must both be length n_samples — interpolate the shift curve
    xp = np.linspace(0, n_samples - 1, n_anchors)
    shift_curve = np.interp(np.arange(n_samples), xp, shifts)

    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)

    warped = np.stack([
        np.interp(new_idx, old_idx, signal[i])
        for i in range(n_leads)
    ], axis=0)
    return warped.astype(signal.dtype)

'''

patched = src[:old_fn_start] + new_fn + src[old_fn_end:]

with open(dataset_path, 'w') as f:
    f.write(patched)

print("✓ _time_warp patched successfully")

# Force Python to reload the module so the running session picks up the fix
import importlib, sys
for mod_name in list(sys.modules.keys()):
    if 'ecg_model' in mod_name:
        del sys.modules[mod_name]

import ecg_model
print("✓ ecg_model reloaded")

In [ ]:
# ── FINAL FIX — in-memory only, no disk writes, no module reloads ────────
import sys, numpy as np

# 1. Remove the np.interp diagnostic trap if it's still installed
if np.interp.__name__ == '_trap':
    from numpy.lib._function_base_impl import interp as _real_interp
    np.interp = _real_interp
    print("✓ np.interp trap removed")

# 2. Grab the live ecg_model.dataset module (do NOT delete from sys.modules)
import ecg_model.dataset
D = sys.modules['ecg_model.dataset']

# 3. Correct _time_warp: xp and fp are both length n_samples, always
def _time_warp_fixed(sig, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = sig.shape
    n_anchors   = 6
    xp_anchors  = np.linspace(0, n_samples - 1, n_anchors)
    shifts      = rng.uniform(-max_warp * n_samples,
                               max_warp * n_samples, n_anchors)
    shift_curve = np.interp(np.arange(n_samples), xp_anchors, shifts)
    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)
    out = np.empty_like(sig, dtype=np.float64)
    for i in range(n_leads):
        out[i] = np.interp(new_idx, old_idx, sig[i])
    return out.astype(sig.dtype)

D._time_warp = _time_warp_fixed
print(f"✓ patched _time_warp in {D.__file__}")

# 4. End-to-end smoke test through augment_signal (which is what training calls)
sig  = np.random.randn(12, 5000).astype(np.float32)
rng  = np.random.default_rng(0)
for _ in range(20):
    out = D.augment_signal(sig, fs=500)
    assert out.shape == sig.shape
print("✓ 20 augmentations passed — no length mismatch")

# 5. Run training. DataLoader workers fork from this process and inherit
#    the patched binding, so worker 0/1 will see _time_warp_fixed.
from ecg_model.train import train as run_training
trained_model = run_training(cfg)

In [ ]:
# ── Kill multiprocessing, re-apply fix, retry ─────────────────────────────
import sys, numpy as np, time, torch

# Re-apply the _time_warp fix (kernel state after an interrupt can be dirty)
import ecg_model.dataset
D = sys.modules['ecg_model.dataset']

def _time_warp_fixed(sig, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = sig.shape
    xp_a  = np.linspace(0, n_samples - 1, 6)
    shifts = rng.uniform(-0.05 * n_samples, 0.05 * n_samples, 6)
    shift_curve = np.interp(np.arange(n_samples), xp_a, shifts)
    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)
    out = np.empty_like(sig, dtype=np.float64)
    for i in range(n_leads):
        out[i] = np.interp(new_idx, old_idx, sig[i])
    return out.astype(sig.dtype)

D._time_warp = _time_warp_fixed
print("✓ _time_warp re-patched")

# Disable multiprocessing — this is the actual unblocker
cfg.num_workers = 0
print(f"✓ num_workers set to {cfg.num_workers}")

# Quick pipeline-health check: can we pull 3 batches in < 30s?
from torch.utils.data import DataLoader
from ecg_model.dataset import load_ptbxl
train_ds, _, _ = load_ptbxl(cfg.data_dir)
probe = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)

t0 = time.time()
for i, (sigs, labels) in enumerate(probe):
    dt = time.time() - t0
    print(f"  batch {i}: signals={tuple(sigs.shape)}  labels={tuple(labels.shape)}  total={dt:.1f}s")
    if i >= 2:
        break
del probe
print(f"✓ Data pipeline is healthy — ~{dt/3:.1f}s per batch")

# Go
from ecg_model.train import train as run_training
trained_model = run_training(cfg)

In [ ]:
# ── Clear VRAM + memory-safe smoke test ───────────────────────────────────
import gc, torch, sys, numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# 1. Free everything from previous runs
if 'model' in dir():
    del model
if 'trained_model' in dir():
    del trained_model
if 'optim' in dir():
    del optim
gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.mem_get_info()[0] / 1e9
print(f"✓ VRAM cleared — {free:.1f} GB free now")

# 2. Re-apply _time_warp fix
import ecg_model.dataset as D
def _time_warp_fixed(sig, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = sig.shape
    xp = np.linspace(0, n_samples - 1, 6)
    shifts = rng.uniform(-0.05 * n_samples, 0.05 * n_samples, 6)
    sc = np.interp(np.arange(n_samples), xp, shifts)
    old = np.arange(n_samples, dtype=np.float64)
    new = np.clip(old + sc, 0, n_samples - 1)
    out = np.empty_like(sig, dtype=np.float64)
    for i in range(n_leads): out[i] = np.interp(new, old, sig[i])
    return out.astype(sig.dtype)
D._time_warp = _time_warp_fixed

# 3. Data
from ecg_model.dataset import load_ptbxl
from ecg_model.model   import build_model, ModelConfig

BATCH_SIZE  = 32     # safe for T4 with 15 GB
SUBSET_FRAC = 0.15
EPOCHS      = 5

train_ds, val_ds, _ = load_ptbxl(cfg.data_dir)
n_sub     = int(len(train_ds) * SUBSET_FRAC)
idx       = np.random.default_rng(42).choice(len(train_ds), n_sub, replace=False)
train_sub = Subset(train_ds, idx)

train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 4. Fresh model + AMP scaler (mixed precision — halves VRAM usage)
device  = torch.device('cuda')
model   = build_model(ModelConfig(n_samples=5000), device='cuda')
optim   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler  = torch.cuda.amp.GradScaler()
crit    = nn.BCEWithLogitsLoss()

used = torch.cuda.memory_allocated() / 1e9
print(f"✓ Model loaded — {used:.2f} GB used, "
      f"{torch.cuda.mem_get_info()[0]/1e9:.2f} GB free")
print(f"Training on {n_sub} samples | {len(train_loader)} batches/epoch | batch={BATCH_SIZE}")
print(f"\n{'Epoch':>5}  {'Train-Loss':>10}  {'Val-Loss':>9}  {'Val-AUC':>8}")
print("-" * 45)

from sklearn.metrics import roc_auc_score

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total_loss, n_batches = 0.0, 0
    bar = tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS} [train]", leave=False)
    for sigs, labels in bar:
        sigs, labels = sigs.to(device), labels.to(device)
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():           # mixed precision
            loss = crit(model(sigs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()
        total_loss += loss.item(); n_batches += 1
        bar.set_postfix(loss=f"{loss.item():.4f}",
                        mem=f"{torch.cuda.memory_allocated()/1e9:.1f}GB")

    # Validate
    model.eval()
    val_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for sigs, labels in tqdm(val_loader, desc=f"Ep {epoch}/{EPOCHS} [val]  ", leave=False):
            sigs, labels = sigs.to(device), labels.to(device)
            logits = model(sigs)
            val_loss += crit(logits, labels).item()
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    try:    auc = roc_auc_score(all_labels, all_probs, average='macro')
    except: auc = float('nan')

    print(f"{epoch:>5}  {total_loss/n_batches:>10.4f}  "
          f"{val_loss/len(val_loader):>9.4f}  {auc:>8.4f}")

print("\n✓ Smoke-test done — if loss is falling and AUC > 0.6, model is working.")

In [ ]:
# ── Full training run with fixed AUC + working config ─────────────────────
import gc, torch, sys, numpy as np, warnings
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

# Clean up smoke-test model
del model, optim, scaler
gc.collect(); torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Re-apply fix
import ecg_model.dataset as D
def _time_warp_fixed(sig, rng, fs=500, max_warp=0.05):
    n_leads, n_samples = sig.shape
    xp = np.linspace(0, n_samples - 1, 6)
    shifts = rng.uniform(-0.05 * n_samples, 0.05 * n_samples, 6)
    sc = np.interp(np.arange(n_samples), xp, shifts)
    old = np.arange(n_samples, dtype=np.float64)
    new = np.clip(old + sc, 0, n_samples - 1)
    out = np.empty_like(sig, dtype=np.float64)
    for i in range(n_leads): out[i] = np.interp(new, old, sig[i])
    return out.astype(sig.dtype)
D._time_warp = _time_warp_fixed

# Data — full dataset this time
from ecg_model.dataset import load_ptbxl
from ecg_model.model   import build_model, ModelConfig

BATCH_SIZE = 32
EPOCHS     = 50
PATIENCE   = 7

train_ds, val_ds, _ = load_ptbxl(cfg.data_dir)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

device  = torch.device('cuda')
model   = build_model(ModelConfig(n_samples=5000), device='cuda')
optim   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=1e-6)
scaler  = torch.amp.GradScaler('cuda')
crit    = nn.BCEWithLogitsLoss()

def compute_auc(probs, labels):
    """Macro AUC ignoring classes with only one label present."""
    aucs = []
    for c in range(labels.shape[1]):
        if len(np.unique(labels[:, c])) < 2:
            continue   # skip Bradycardia / rare classes
        aucs.append(roc_auc_score(labels[:, c], probs[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')

best_auc, no_improve = 0.0, 0
print(f"\n{'Ep':>3}  {'Train-L':>8}  {'Val-L':>7}  {'Val-AUC':>8}  {'LR':>8}")
print("-" * 48)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    tloss, n = 0.0, 0
    bar = tqdm(train_loader, desc=f"Ep {epoch:02d}/{EPOCHS} train", leave=False,
               bar_format="{l_bar}{bar:20}{r_bar}")
    for sigs, labels in bar:
        sigs, labels = sigs.to(device), labels.to(device)
        optim.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = crit(model(sigs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim); scaler.update()
        tloss += loss.item(); n += 1
        bar.set_postfix(loss=f"{loss.item():.3f}",
                        mem=f"{torch.cuda.memory_allocated()/1e9:.1f}G")
    sched.step()

    # ── Validate ──
    model.eval()
    vloss, probs_all, labels_all = 0.0, [], []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for sigs, labels in val_loader:
            sigs, labels = sigs.to(device), labels.to(device)
            logits = model(sigs)
            vloss += crit(logits, labels).item()
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    probs_all  = np.concatenate(probs_all)
    labels_all = np.concatenate(labels_all)
    auc = compute_auc(probs_all, labels_all)
    lr  = sched.get_last_lr()[0]

    print(f"{epoch:>3}  {tloss/n:>8.4f}  {vloss/len(val_loader):>7.4f}"
          f"  {auc:>8.4f}  {lr:>8.2e}")

    # ── Checkpoint + early stopping ──
    if auc > best_auc + 1e-4:
        best_auc, no_improve = auc, 0
        torch.save(model.state_dict(), f"{cfg.checkpoint_dir}/best_model.pt")
        print(f"           ↑ saved (AUC {auc:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

print(f"\n✓ Training complete — best val AUC: {best_auc:.4f}")
print(f"  Checkpoint: {cfg.checkpoint_dir}/best_model.pt")

## 9. Plot training curves

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path
from ecg_model.evaluate import plot_training_history

history_path = f'{LOG_DIR}/history.json'
plot_training_history(history_path, EVAL_DIR)

with open(history_path) as f:
    history = json.load(f)

best = max(history, key=lambda h: h['val_auc'])
print(f'Best epoch:   {best["epoch"]}')
print(f'Best val AUC: {best["val_auc"]:.4f}')
print(f'Val loss:     {best["val_loss"]:.4f}')

# Show training curves inline.
from PIL import Image
img = Image.open(f'{EVAL_DIR}/training_curves.png')
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 10. Evaluate on held-out test set

In [ ]:
from ecg_model.evaluate import evaluate
from pathlib import Path

calibrated_path = f'{CHECKPOINT_DIR}/calibrated_model.pt'

summary = evaluate(
    checkpoint_path = calibrated_path,
    data_dir        = DATA_DIR,
    output_dir      = EVAL_DIR,
    batch_size      = 64,
    num_workers     = 2,
)

## 11. View calibration & ROC plots

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for fname, title in [
    ('roc_curves.png', 'ROC Curves'),
    ('calibration_curves.png', 'Calibration Curves (Reliability Diagrams)'),
    ('class_distribution.png', 'Class Distribution'),
]:
    path = f'{EVAL_DIR}/{fname}'
    try:
        img = Image.open(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.tight_layout()
        plt.show()
    except FileNotFoundError:
        print(f'{fname} not found yet — run the evaluate cell first')

## 12. Quick inference test — single ECG

In [ ]:
import torch
from ecg_model.model import load_checkpoint, CONDITION_NAMES
from ecg_model.dataset import load_ptbxl

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_checkpoint(f'{CHECKPOINT_DIR}/calibrated_model.pt', device=str(device))
model.eval()

# Grab one record from the test set.
_, _, test_ds = load_ptbxl(DATA_DIR)
signal, true_labels = test_ds[0]

with torch.no_grad():
    probs = model.predict_proba(signal.unsqueeze(0).to(device))[0].cpu().numpy()

print(f'Single-record inference results:')
print(f'{"Condition":<44} Prob    True')
print('-' * 60)
for cond, prob, truth in zip(CONDITION_NAMES, probs, true_labels):
    bar = '█' * int(prob * 20) + '░' * (20 - int(prob * 20))
    marker = '✓' if truth == 1 else ' '
    print(f'  {cond:<42} [{bar}] {prob:.3f}  {marker}')

## 13. Copy final checkpoint for Phase 4 (API)
The API (Phase 4) will load `calibrated_model.pt` at startup. Copy it to a
stable location in Drive that the FastAPI Docker container can access.

In [ ]:
import shutil
from pathlib import Path

src = f'{CHECKPOINT_DIR}/calibrated_model.pt'
dst = f'{BASE_DIR}/model_release/calibrated_model.pt'
Path(dst).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
print(f'Checkpoint copied to: {dst}')
print(f'File size: {Path(dst).stat().st_size / 1e6:.1f} MB')
print()
print('Next steps:')
print('  1. Download calibrated_model.pt from Drive to your local machine')
print('  2. Place it in the phase4/api/model/ directory')
print('  3. Proceed to Phase 4 (FastAPI Cloud REST API)')

## Troubleshooting

**CUDA out of memory**
Reduce `batch_size` from 64 to 32 in the TrainConfig cell, then re-run.

**Session disconnected during training**
Re-run cells 1, 2, 3, 8 (the training cell). The best checkpoint is already
in Drive. Training will continue from epoch 1, but early stopping will
kick in once the new run can't beat the saved best.
For proper resume, add `torch.load(CHECKPOINT_DIR/best_model.pt)` and
restore optimizer state — a future improvement.

**PTB-XL download stalls**
Re-run cells 4a and 4b. The Kaggle download is resumable — already-extracted files in Drive are skipped automatically. Make sure your `kaggle.json` is valid and your Kaggle account has accepted the dataset terms at https://www.kaggle.com/datasets/khyeh0719/ptb-xl-dataset

**Low AUC on a specific condition**
Check the class distribution plot — very rare conditions (< 2%) need a
higher `pos_weight_cap` or focal loss. ST Elevation and PVC in particular
are underrepresented in PTB-XL.